# bench-tabular-stroke: Rare Early Recurrent Stroke Detection

This notebook benchmarks a difficult, realistic tabular clinical-trial use case: ranking patients by risk of a recurrent stroke within 14 days of randomization in the International Stroke Trial (IST). The endpoint occurs in only about 4% of participants. The task is useful for trial monitoring and risk-adjusted analysis, but this notebook is a methodological benchmark, not a diagnostic device or evidence for excluding participants.

The original placebo super-responder idea was changed because no pharmaceutical-scale, openly downloadable participant-level placebo dataset with the necessary response labels was found. Project Data Sphere and Vivli distribute many de-identified trials through controlled request workflows, not unrestricted direct downloads. IST provides a fully anonymous, directly downloadable 19,435-participant table under an open attribution license, making the benchmark reproducible.

The notebook follows the finance and climate benchmarks: explicit temporal splits, train-only preprocessing, a classical and foundation-model zoo, a skrub ablation, rare-event augmentation benchmarks, and repeated improvement loops with held-out evaluation.

## 1. Setup and execution constraints

The run must be launched with `CUDA_VISIBLE_DEVICES=1`. Inside the process, physical GPU 1 is remapped to `cuda:0`. CPU libraries and every estimator are restricted to one thread, which is far below 95% of this 128-core VM.

In [ ]:
SEED = 42
RUN_FOUNDATION_MODELS = True
FOUNDATION_CONTEXT = 2048
FOUNDATION_CHUNK = 2048

In [ ]:
import os

THREAD_VARS = ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]
for name in THREAD_VARS:
    os.environ.setdefault(name, "1")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")
os.environ.setdefault("WANDB_MODE", "offline")
assert os.environ.get("CUDA_VISIBLE_DEVICES") == "1", "Launch with CUDA_VISIBLE_DEVICES=1"

import copy
import gc
import hashlib
import json
import math
import time
import urllib.request
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, brier_score_loss,
    f1_score, fbeta_score, precision_recall_curve, precision_score,
    recall_score, roc_auc_score, RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from skrub import TableVectorizer

from ml_pipeline.data_augmentation.augmentations import AUGMENTATION_REGISTRY
from ml_pipeline.pipelines_torch.models import MODEL_REGISTRY
from ml_pipeline.utils.wandb_utils import finish_run, init_wandb_run, log_summary

warnings.filterwarnings("ignore")
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass
assert torch.cuda.is_available() and torch.cuda.device_count() == 1
DEVICE = "cuda:0"
print({
    "visible_physical_gpu": os.environ["CUDA_VISIBLE_DEVICES"],
    "torch_device": DEVICE,
    "gpu_name": torch.cuda.get_device_name(0),
    "cpu_threads": {name: os.environ[name] for name in THREAD_VARS},
})

## 2. Data provenance and endpoint

The [University of Edinburgh DataShare record](https://datashare.ed.ac.uk/items/91709c88-d913-45d8-8019-380027133d54) provides the corrected IST participant table and variable dictionary by direct download. The [ODC Attribution license](https://opendatacommons.org/licenses/by/1-0/) permits use and adaptation with attribution. The accompanying [database paper](https://datashare.ed.ac.uk/bitstream/handle/10283/124/IST_database_paper.pdf) describes the randomized aspirin and heparin trial, near-complete follow-up, and anonymization.

`STRK14` marks any recurrent stroke recorded within 14 days. Predictors are restricted to information available at randomization: demographics, presentation delay, consciousness, prior medication, systolic blood pressure, neurological deficits, stroke subtype, country, and randomization timing. Hospital identifiers, assigned treatment, event components, follow-up fields, and derived outcomes are excluded. The label is an outcome association, not a causal treatment effect.

In [ ]:
DATA_DIR = Path("ml_pipeline/data/tabular_stroke")
RESULTS_DIR = Path("ml_pipeline/results/tabular_stroke")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_URL = "https://datashare.ed.ac.uk/server/api/core/bitstreams/eca52b56-6d1b-47b3-814a-0b7089f8c870/content"
DATA_SHA256 = "a350aed2329bbcde23c05caaa96b9275b44b017adecc2fd02231d590454695d5"
DATA_PATH = DATA_DIR / "IST_corrected.csv"
if not DATA_PATH.exists():
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
actual_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
assert actual_sha256 == DATA_SHA256, f"Unexpected IST file checksum: {actual_sha256}"

df = pd.read_csv(DATA_PATH, encoding="latin1", low_memory=False)
assert df.shape == (19435, 112)
assert set(df["STRK14"].unique()) == {0, 1}
df["randomization_year"] = 1900 + df["RDATE"].str[-2:].astype(int)
print(f"Loaded {len(df):,} participants and {df.shape[1] - 1} source columns")
print(f"Early recurrent strokes: {df['STRK14'].sum():,} ({df['STRK14'].mean():.2%})")

In [ ]:
BASE_FEATURES = [
    "RDELAY", "RCONSC", "SEX", "AGE", "RSLEEP", "RATRIAL",
    "RCT", "RVISINF", "RHEP24", "RASP3", "RSBP",
    *[f"RDEF{i}" for i in range(1, 9)],
    "STYPE", "COUNTRY", "HOURLOCAL", "DAYLOCAL",
]
TARGET = "STRK14"
assert len(BASE_FEATURES) == 23

feature_audit = pd.DataFrame({
    "dtype": df[BASE_FEATURES].dtypes.astype(str),
    "missing": df[BASE_FEATURES].isna().sum(),
    "unique": df[BASE_FEATURES].nunique(dropna=True),
}).sort_values(["missing", "unique"], ascending=False)
display(feature_audit)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x=TARGET, ax=axes[0])
axes[0].set_title("Severe class imbalance")
by_year = df.groupby("randomization_year")[TARGET].agg(["size", "sum", "mean"]).reset_index()
sns.lineplot(data=by_year, x="randomization_year", y="mean", marker="o", ax=axes[1])
axes[1].axhline(df[TARGET].mean(), color="gray", linestyle="--", label="overall")
axes[1].set_ylabel("event rate")
axes[1].set_title("Endpoint prevalence drifts over calendar time")
axes[1].legend()
plt.tight_layout()
plt.show()
display(by_year.style.format({"mean": "{:.2%}"}))

## 3. Temporal split and leakage controls

A random split would mix enrollment eras, sites, and changing clinical practice. Training uses 1991 to 1994, validation uses 1995, and the final test is 1996. Preprocessing, resampling, early stopping, threshold selection, and model selection never fit on test rows. The falling endpoint rate makes calibration shift part of the benchmark.

In [ ]:
split_masks = {
    "train": df["randomization_year"] <= 1994,
    "validation": df["randomization_year"] == 1995,
    "test": df["randomization_year"] == 1996,
}
assert sum(mask.sum() for mask in split_masks.values()) == len(df)

X_raw = {name: df.loc[mask, BASE_FEATURES].copy() for name, mask in split_masks.items()}
y = {name: df.loc[mask, TARGET].to_numpy(dtype=np.int64) for name, mask in split_masks.items()}
split_summary = pd.DataFrame([
    {"split": name, "rows": len(y[name]), "events": int(y[name].sum()), "event_rate": y[name].mean()}
    for name in split_masks
])
display(split_summary.style.format({"event_rate": "{:.2%}"}))
assert split_summary.set_index("split").loc["test", "events"] >= 50

## 4. skrub preprocessing

`TableVectorizer` learns numeric passthrough, missing-value handling, and low-cardinality encoding from the training period only. Unseen categories are ignored safely. The dense 79-column representation is small enough for both classical models and the tabular foundation models.

In [ ]:
preprocess_start = time.perf_counter()
vectorizer = TableVectorizer(n_jobs=1)
X = {"train": np.asarray(vectorizer.fit_transform(X_raw["train"]), dtype=np.float32)}
X["validation"] = np.asarray(vectorizer.transform(X_raw["validation"]), dtype=np.float32)
X["test"] = np.asarray(vectorizer.transform(X_raw["test"]), dtype=np.float32)
skrub_seconds = time.perf_counter() - preprocess_start
assert all(np.isfinite(values).all() for values in X.values())
print({name: values.shape for name, values in X.items()})
print(f"skrub fit plus three transforms: {skrub_seconds:.3f} s")

## 5. Metrics and model zoo

Average precision, also called PR-AUC here, is primary because a prevalence-only ranker scores approximately the event rate. ROC-AUC measures pairwise ranking. Brier score measures probability error. Recall at the top 10% asks how many events are captured if monitoring capacity is limited to one patient in ten. Precision, recall, F1, F2, and balanced accuracy initially use the conventional 0.5 threshold; a later loop learns a decision threshold from validation only.

Classical estimators use the repository model registry. The MLP uses the registry architecture with a small explicit training loop on GPU. TabICLv2 and TabFM use representative 4,096-row training contexts and chunked GPU inference so that foundation-model comparison remains reproducible on 48 GB hardware.

In [ ]:
def metric_row(model_name, y_true, probability, threshold=0.5, fit_seconds=np.nan, predict_seconds=np.nan):
    probability = np.asarray(probability, dtype=float)
    prediction = probability >= threshold
    budget = max(1, math.ceil(0.10 * len(y_true)))
    top_idx = np.argsort(-probability)[:budget]
    return {
        "model": model_name,
        "roc_auc": roc_auc_score(y_true, probability),
        "pr_auc": average_precision_score(y_true, probability),
        "brier": brier_score_loss(y_true, probability),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "f2": fbeta_score(y_true, prediction, beta=2, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "recall_at_10pct": y_true[top_idx].sum() / max(1, y_true.sum()),
        "threshold": threshold,
        "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds,
    }

def chunked_probability(model, values, chunk_size=FOUNDATION_CHUNK):
    parts = []
    for start in range(0, len(values), chunk_size):
        parts.append(np.asarray(model.predict_proba(values[start:start + chunk_size]))[:, 1])
    return np.concatenate(parts)

def train_mlp(X_train, y_train, X_validation, y_validation, epochs=35):
    model = MODEL_REGISTRY["mlp_classifier"](input_dim=X_train.shape[1], num_classes=2, dropout=0.25).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-4)
    criterion = torch.nn.CrossEntropyLoss()
    dataset = torch.utils.data.TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
    loader = torch.utils.data.DataLoader(dataset, batch_size=256, shuffle=True, generator=torch.Generator().manual_seed(SEED))
    validation_tensor = torch.from_numpy(X_validation).to(DEVICE)
    best_ap, best_state, patience = -np.inf, None, 7
    stale = 0
    for epoch in range(epochs):
        model.train()
        for batch_x, batch_y in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_x.to(DEVICE)), batch_y.to(DEVICE))
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.inference_mode():
            val_probability = torch.softmax(model(validation_tensor), dim=1)[:, 1].cpu().numpy()
        val_ap = average_precision_score(y_validation, val_probability)
        if val_ap > best_ap + 1e-5:
            best_ap, best_state, stale = val_ap, copy.deepcopy(model.state_dict()), 0
        else:
            stale += 1
            if stale >= patience:
                break
    model.load_state_dict(best_state)
    return model, epoch + 1, best_ap

def torch_probability(model, values):
    model.eval()
    parts = []
    with torch.inference_mode():
        for start in range(0, len(values), 1024):
            logits = model(torch.from_numpy(values[start:start + 1024]).to(DEVICE))
            parts.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
    return np.concatenate(parts)

In [ ]:
classical_specs = {
    "logreg": ("logreg", dict(max_iter=3000, C=1.0, n_jobs=1, random_state=SEED)),
    "random_forest": ("random_forest_classifier", dict(n_estimators=300, min_samples_leaf=5, max_features="sqrt", n_jobs=1, random_state=SEED)),
    "xgboost": ("xgboost_classifier", dict(n_estimators=350, max_depth=4, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, n_jobs=1, random_state=SEED, eval_metric="logloss")),
    "lightgbm": ("lightgbm_classifier", dict(n_estimators=350, num_leaves=15, learning_rate=0.03, min_child_samples=40, n_jobs=1, random_state=SEED, verbosity=-1)),
    "catboost": ("catboost_classifier", dict(iterations=350, depth=6, learning_rate=0.03, thread_count=1, random_seed=SEED, verbose=False, allow_writing_files=False)),
}

predictions = {"validation": {}, "test": {}}
timing = {}
errors = {}

start = time.perf_counter()
dummy = DummyClassifier(strategy="prior").fit(X["train"], y["train"])
timing["dummy"] = (time.perf_counter() - start, 0.0)
for split in ["validation", "test"]:
    predictions[split]["dummy"] = dummy.predict_proba(X[split])[:, 1]

for name, (registry_key, params) in classical_specs.items():
    try:
        if registry_key == "logreg":
            model = LogisticRegression(**params)
        else:
            model = MODEL_REGISTRY[registry_key](**params)
        start = time.perf_counter()
        model.fit(X["train"], y["train"])
        fit_seconds = time.perf_counter() - start
        start = time.perf_counter()
        for split in ["validation", "test"]:
            predictions[split][name] = np.asarray(model.predict_proba(X[split]))[:, 1]
        predict_seconds = time.perf_counter() - start
        timing[name] = (fit_seconds, predict_seconds)
        run = init_wandb_run(name=f"stroke-{name}", group="bench-tabular-stroke", config=params, tags=["tabular", "clinical", "rare-event"])
        log_summary(run, metric_row(name, y["test"], predictions["test"][name]))
        finish_run(run)
    except Exception as exc:
        errors[name] = repr(exc)
        print(f"{name} failed: {exc}")

In [ ]:
try:
    start = time.perf_counter()
    mlp, mlp_epochs, mlp_best_val_ap = train_mlp(X["train"], y["train"], X["validation"], y["validation"])
    fit_seconds = time.perf_counter() - start
    start = time.perf_counter()
    for split in ["validation", "test"]:
        predictions[split]["mlp"] = torch_probability(mlp, X[split])
    timing["mlp"] = (fit_seconds, time.perf_counter() - start)
    print(f"MLP stopped after {mlp_epochs} epochs; best validation PR-AUC={mlp_best_val_ap:.4f}")
except Exception as exc:
    errors["mlp"] = repr(exc)
    print(f"mlp failed: {exc}")
finally:
    if "mlp" in locals():
        del mlp
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
if RUN_FOUNDATION_MODELS:
    context_idx, _ = train_test_split(
        np.arange(len(y["train"])), train_size=FOUNDATION_CONTEXT,
        stratify=y["train"], random_state=SEED,
    )
    X_context, y_context = X["train"][context_idx], y["train"][context_idx]
    foundation_specs = {
        "tabicl_v2": ("tabicl_classifier", dict(n_estimators=2, batch_size=1, device=DEVICE, use_fa3=False, offload_mode="gpu", n_jobs=1, random_state=SEED)),
        "tabfm": ("tabfm_classifier", dict(device=DEVICE, n_estimators=2, batch_size=1, max_num_rows=FOUNDATION_CONTEXT, random_state=SEED)),
    }
    for name, (registry_key, params) in foundation_specs.items():
        try:
            start = time.perf_counter()
            model = MODEL_REGISTRY[registry_key](**params)
            model.fit(X_context, y_context)
            fit_seconds = time.perf_counter() - start
            start = time.perf_counter()
            for split in ["validation", "test"]:
                predictions[split][name] = chunked_probability(model, X[split])
            timing[name] = (fit_seconds, time.perf_counter() - start)
        except Exception as exc:
            errors[name] = repr(exc)
            print(f"{name} failed: {exc}")
        finally:
            if "model" in locals():
                del model
            gc.collect()
            torch.cuda.empty_cache()

print("Completed models:", sorted(predictions["test"]))
print("Errors:", errors)

## 6. Baseline results

In [ ]:
validation_results = pd.DataFrame([
    metric_row(name, y["validation"], probability, fit_seconds=timing.get(name, (np.nan, np.nan))[0], predict_seconds=timing.get(name, (np.nan, np.nan))[1])
    for name, probability in predictions["validation"].items()
]).sort_values("pr_auc", ascending=False)
test_results = pd.DataFrame([
    metric_row(name, y["test"], probability, fit_seconds=timing.get(name, (np.nan, np.nan))[0], predict_seconds=timing.get(name, (np.nan, np.nan))[1])
    for name, probability in predictions["test"].items()
]).sort_values("pr_auc", ascending=False)
validation_results.to_csv(RESULTS_DIR / "baseline_validation_metrics.csv", index=False)
test_results.to_csv(RESULTS_DIR / "baseline_test_metrics.csv", index=False)
metric_columns = ["model", "roc_auc", "pr_auc", "brier", "recall_at_10pct", "precision", "recall", "f2", "fit_seconds", "predict_seconds"]
print("Validation, sorted by PR-AUC")
display(validation_results[metric_columns].style.format(precision=4))
print("Untouched 1996 test, sorted by PR-AUC")
display(test_results[metric_columns].style.format(precision=4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for name, probability in predictions["test"].items():
    PrecisionRecallDisplay.from_predictions(y["test"], probability, name=name, ax=axes[0])
    RocCurveDisplay.from_predictions(y["test"], probability, name=name, ax=axes[1])
    fraction_positive, mean_predicted = calibration_curve(y["test"], probability, n_bins=8, strategy="quantile")
    axes[2].plot(mean_predicted, fraction_positive, marker="o", label=name)
axes[0].axhline(y["test"].mean(), color="black", linestyle="--", label="prevalence")
axes[0].set_title("1996 precision-recall curves")
axes[1].set_title("1996 ROC curves")
axes[2].plot([0, 0.25], [0, 0.25], color="black", linestyle="--")
axes[2].set(xlabel="mean predicted probability", ylabel="observed event rate", title="Calibration under temporal shift")
axes[2].legend(fontsize=7)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "baseline_curves.png", dpi=160, bbox_inches="tight")
plt.show()

### Baseline reading

Validation favors the two zero-shot foundation models: TabFM reaches PR-AUC 0.0529 and TabICLv2 0.0528, ahead of the MLP at 0.0521 and the random forest at 0.0481. All beat the 1995 prevalence baseline of 0.0369. The 1996 ranking is unstable: the MLP has the best descriptive test PR-AUC at 0.0449 and the random forest follows at 0.0423, while TabFM and TabICLv2 fall to 0.0374 and 0.0370. This is evidence of temporal distribution shift, not a reason to select the MLP after seeing test labels.

The random forest has the strongest baseline test ROC-AUC at 0.5642. The dummy has the lowest Brier score because predicting the low prevalence is hard to beat in squared probability error. Every model has zero recall at the conventional 0.5 threshold, showing why discrimination, calibration, and an explicit operating policy must be reported separately.

## 7. Ablation: does skrub help?

The comparison below keeps logistic regression fixed. The manual alternative median-imputes and standardizes numeric fields, then most-frequent-imputes and one-hot encodes categorical fields. This measures preprocessing time, output width, and predictive metrics. It does not claim a general package ranking from one small dataset.

In [ ]:
numeric_columns = X_raw["train"].select_dtypes(exclude="object").columns.tolist()
categorical_columns = [column for column in BASE_FEATURES if column not in numeric_columns]
manual_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_columns),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)),
    ]), categorical_columns),
])
start = time.perf_counter()
X_manual = {"train": np.asarray(manual_preprocessor.fit_transform(X_raw["train"]), dtype=np.float32)}
for split in ["validation", "test"]:
    X_manual[split] = np.asarray(manual_preprocessor.transform(X_raw[split]), dtype=np.float32)
manual_seconds = time.perf_counter() - start

ablation_rows = []
for name, values, prep_seconds in [("skrub", X, skrub_seconds), ("manual", X_manual, manual_seconds)]:
    model = LogisticRegression(max_iter=3000, C=1.0, n_jobs=1, random_state=SEED)
    start = time.perf_counter()
    model.fit(values["train"], y["train"])
    fit_seconds = time.perf_counter() - start
    for split in ["validation", "test"]:
        probability = model.predict_proba(values[split])[:, 1]
        row = metric_row(name, y[split], probability, fit_seconds=fit_seconds)
        row.update({"split": split, "features": values["train"].shape[1], "preprocess_seconds": prep_seconds})
        ablation_rows.append(row)
ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(RESULTS_DIR / "skrub_ablation.csv", index=False)
display(ablation_df[["split", "model", "features", "preprocess_seconds", "fit_seconds", "roc_auc", "pr_auc", "brier"]].style.format(precision=4))

### Ablation result

For logistic regression, skrub produces 79 features and slightly improves test ROC-AUC from 0.5333 to 0.5365 and PR-AUC from 0.0386 to 0.0395. It is not computationally faster here: preprocessing takes 1.396 seconds versus 0.165 seconds manually, and the unscaled skrub representation also makes logistic fitting slower. On this small, low-cardinality table, skrub's value is concise and safe mixed-type handling plus a small predictive gain, not raw speed.

## 8. Improvement loop 1: train-only rare-event augmentation

Hypothesis: generating minority examples can improve ranking when only 478 positive training rows exist. SMOTE, Borderline-SMOTE, and ADASYN come from the shared augmentation registry and are applied only after fitting `TableVectorizer` on training data. `max_factor=8` or `4` means the final minority count targets one eighth or one quarter of the majority count before the registry's Tomek cleaning.

Caution: one-hot coordinates become fractional under interpolation. This is mathematically valid input to the estimator but does not necessarily represent a real patient, so gains must be validated rather than assumed.

In [ ]:
augmentation_specs = [
    ("none", {}),
    ("smote_factor8", {"registry": "smote", "max_factor": 8.0}),
    ("smote_factor4", {"registry": "smote", "max_factor": 4.0}),
    ("borderline_factor4", {"registry": "borderline_smote", "max_factor": 4.0}),
    ("adasyn_factor4", {"registry": "adasyn", "max_factor": 4.0}),
]
augmentation_rows = []
augmentation_predictions = {}
for name, spec in augmentation_specs:
    start = time.perf_counter()
    if name == "none":
        X_resampled, y_resampled = X["train"], y["train"]
    else:
        X_resampled, y_resampled = AUGMENTATION_REGISTRY[spec["registry"]](
            X["train"], y["train"], random_state=SEED, max_factor=spec["max_factor"]
        )
    augmentation_seconds = time.perf_counter() - start
    model = RandomForestClassifier(
        n_estimators=250, min_samples_leaf=5, max_features="sqrt",
        n_jobs=1, random_state=SEED,
    )
    start = time.perf_counter()
    model.fit(X_resampled, y_resampled)
    fit_seconds = time.perf_counter() - start
    for split in ["validation", "test"]:
        probability = model.predict_proba(X[split])[:, 1]
        if split == "test":
            augmentation_predictions[name] = probability
        row = metric_row(name, y[split], probability, fit_seconds=fit_seconds)
        row.update({
            "split": split, "train_rows": len(y_resampled),
            "train_events": int(np.sum(y_resampled == 1)),
            "augmentation_seconds": augmentation_seconds,
        })
        augmentation_rows.append(row)
augmentation_df = pd.DataFrame(augmentation_rows)
augmentation_df.to_csv(RESULTS_DIR / "augmentation_metrics.csv", index=False)
display(augmentation_df[["split", "model", "train_rows", "train_events", "augmentation_seconds", "fit_seconds", "roc_auc", "pr_auc", "brier", "recall_at_10pct"]].style.format(precision=4))

validation_augmentation = augmentation_df.query("split == 'validation'").sort_values("pr_auc", ascending=False)
best_augmentation_name = validation_augmentation.iloc[0]["model"]
base_augmentation_val = validation_augmentation.set_index("model").loc["none", "pr_auc"]
best_augmentation_val = validation_augmentation.iloc[0]["pr_auc"]
print(f"Validation-selected augmentation: {best_augmentation_name}; PR-AUC delta vs none: {best_augmentation_val - base_augmentation_val:+.4f}")

### Loop 1 result: a small valid improvement

SMOTE with `max_factor=8` is selected on validation: PR-AUC rises from 0.0486 without augmentation to 0.0515. The frozen choice also improves test PR-AUC from 0.0409 to 0.0421 and top-10% recall from 10.3% to 11.5%. More aggressive balancing does not help consistently. ADASYN has the highest descriptive test PR-AUC at 0.0467 but the worst validation PR-AUC in this comparison, so selecting it from test performance would be leakage.

## 9. Improvement loop 2: lower-variance extremely randomized trees

Hypothesis: ordinary forests correlate their trees through greedy split selection. ExtraTrees randomizes split thresholds and uses a larger leaf floor, which may reduce variance in a dataset with few events and modest signal. The configuration is fixed before looking at the 1996 test result and is compared with the identically preprocessed random-forest baseline.

In [ ]:
extra_model = ExtraTreesClassifier(
    n_estimators=500, min_samples_leaf=10, max_features=0.8,
    n_jobs=1, random_state=SEED,
)
start = time.perf_counter()
extra_model.fit(X["train"], y["train"])
extra_fit_seconds = time.perf_counter() - start
extra_predictions = {split: extra_model.predict_proba(X[split])[:, 1] for split in ["validation", "test"]}
extra_rows = pd.DataFrame([
    metric_row("extra_trees", y[split], extra_predictions[split], fit_seconds=extra_fit_seconds) | {"split": split}
    for split in ["validation", "test"]
])
comparison_rows = []
for split, baseline_table in [("validation", validation_results), ("test", test_results)]:
    baseline = baseline_table.set_index("model").loc["random_forest"]
    improved = extra_rows.set_index("split").loc[split]
    comparison_rows.append({
        "split": split,
        "baseline_roc_auc": baseline["roc_auc"],
        "extra_roc_auc": improved["roc_auc"],
        "delta_roc_auc": improved["roc_auc"] - baseline["roc_auc"],
        "baseline_pr_auc": baseline["pr_auc"],
        "extra_pr_auc": improved["pr_auc"],
        "delta_pr_auc": improved["pr_auc"] - baseline["pr_auc"],
    })
extra_comparison = pd.DataFrame(comparison_rows)
extra_comparison.to_csv(RESULTS_DIR / "extra_trees_improvement.csv", index=False)
display(extra_comparison.style.format(precision=4))

### Loop 2 result: mixed, not a primary-metric win

ExtraTrees improves ROC-AUC over the random forest by 0.0057 on validation and 0.0024 on test. Test PR-AUC also rises sharply from 0.0423 to 0.0542, but validation PR-AUC falls from 0.0481 to 0.0464. The result supports the variance-reduction hypothesis for ROC ranking, but it is not counted as a clean PR-AUC improvement because the primary validation metric moved backward.

## 10. Improvement loop 3: validation-selected operating threshold

A 0.5 cutoff is inappropriate when calibrated risks are only a few percent. We select the threshold maximizing F2 on 1995 validation predictions, placing four times as much weight on recall as precision, and then freeze it for 1996. This changes the action policy, not ROC-AUC or PR-AUC. It is valid only for a monitoring workflow whose false-positive cost is compatible with F2.

In [ ]:
def validation_f2_threshold(y_true, probability):
    precision, recall, thresholds = precision_recall_curve(y_true, probability)
    f2 = 5 * precision[:-1] * recall[:-1] / np.maximum(4 * precision[:-1] + recall[:-1], 1e-12)
    return float(thresholds[int(np.nanargmax(f2))])

candidates = {name: predictions["validation"][name] for name in predictions["validation"]}
candidates["extra_trees"] = extra_predictions["validation"]
best_policy_model = max(candidates, key=lambda name: average_precision_score(y["validation"], candidates[name]))
policy_validation_probability = candidates[best_policy_model]
policy_test_probability = extra_predictions["test"] if best_policy_model == "extra_trees" else predictions["test"][best_policy_model]
selected_threshold = validation_f2_threshold(y["validation"], policy_validation_probability)

threshold_rows = []
for split, probability in [("validation", policy_validation_probability), ("test", policy_test_probability)]:
    threshold_rows.append(metric_row(f"{best_policy_model}_at_0.5", y[split], probability, threshold=0.5) | {"split": split})
    threshold_rows.append(metric_row(f"{best_policy_model}_f2_threshold", y[split], probability, threshold=selected_threshold) | {"split": split})
threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(RESULTS_DIR / "threshold_improvement.csv", index=False)
print(f"Validation-selected ranking model: {best_policy_model}")
print(f"Frozen F2 threshold: {selected_threshold:.5f}")
display(threshold_df[["split", "model", "threshold", "precision", "recall", "f1", "f2", "balanced_accuracy"]].style.format(precision=4))

## 11. Summary and limitations

### Loop 3 result: a valid operational improvement with a major cost

TabFM is selected by validation PR-AUC. Its validation-optimal F2 threshold is 0.02438, far below 0.5. Freezing that threshold raises validation recall from 0 to 72.7% and test recall from 0 to 72.4%; test F2 rises from 0 to 0.1546. Precision remains only 3.73%, however, and 1,690 of 2,675 test patients would be flagged to find 63 of 87 events. This policy is useful only when missed events are much costlier than reviewing false alerts. A resource-limited workflow should instead use the reported top-k recall or choose a stricter capacity-constrained threshold.

### Overall conclusion

This is tabular machine learning over 23 mixed numeric and categorical baseline variables. The benchmark is difficult but non-trivial: the best test rankings improve on the 3.25% prevalence baseline, yet model ordering changes across years and absolute precision stays low. Train-only SMOTE provides one small, validation-supported discrimination gain. Validation-only threshold selection provides a second valid improvement in recall and F2, with an explicit alert-volume tradeoff. The central safeguards remain fixed: open anonymous data, baseline-only covariates, calendar-time evaluation, training-only preprocessing and augmentation, validation-only early stopping and thresholding, and an untouched 1996 test period.

Limitations: recurrent stroke is not placebo response; IST was conducted in the 1990s and does not represent modern treatment; only 87 test events make model ordering noisy; country and practice patterns may shift; interpolated one-hot rows are not clinically coherent synthetic patients; no subgroup fairness or external-cohort validation is performed; and association scores must not be used to deny treatment or trial participation.

In [ ]:
summary = {
    "rows": len(df),
    "overall_event_rate": float(df[TARGET].mean()),
    "test_event_rate": float(y["test"].mean()),
    "best_validation_model": validation_results.iloc[0]["model"],
    "best_validation_pr_auc": float(validation_results.iloc[0]["pr_auc"]),
    "best_test_model_descriptive": test_results.iloc[0]["model"],
    "best_test_pr_auc_descriptive": float(test_results.iloc[0]["pr_auc"]),
    "best_augmentation_by_validation": best_augmentation_name,
    "policy_model": best_policy_model,
    "policy_threshold": selected_threshold,
    "errors": errors,
}
(RESULTS_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))